# 🎙️ ElevenLabs SOTA Russian Studio (Flow-Matching DiT + Vocos)
### Ультра-премиум клонирование живого голоса на **русском языке** без роботов и скрежета
---
> * 🔥 **Flow-Matching DiT**: Новейшая диффузионная архитектура (полное отсутствие металлического звука).
> * 🇷🇺 **5,000+ часов русской речи**: Специальная модель `Misha24-10/F5-TTS_RUSSIAN` с идеальной фонетикой.
> * 🔤 **RUAccent**: Автоматическая расстановка русских ударений на лету.
> * 📚 **VoiceLab Library**: Сохраняйте голоса 1 раз — они остаются в выпадающем списке навсегда.
> * 💾 **Google Drive**: Постоянное хранение всех созданных голосов в `MyDrive/ElevenLabs_Voices`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markankamaransheogarat-sys/open-elevenlabs-voice-studio/blob/main/Open_ElevenLabs_Voice_Studio_F5_TTS.ipynb)

## 1. ⚙️ Установка SOTA окружения
Нажмите **Play**. Занимает около 50 секунд.

In [ ]:
# @title ⚙️ 1. Установка SOTA Russian Diffusion Studio (Run Setup)
# Подключение Google Drive для вечного сохранения голосов
try:
    from google.colab import drive
    drive.mount('/content/drive')
    VOICES_DIR = '/content/drive/MyDrive/ElevenLabs_Voices'
    print('💾 Google Drive подключен! Ваши голоса сохраняются в папку ElevenLabs_Voices.')
except Exception:
    VOICES_DIR = '/content/voices'
    print('💾 Работаем локально (голоса будут в /content/voices).')

import os
os.makedirs(VOICES_DIR, exist_ok=True)

!nvidia-smi
print('📦 Установка Flow-Matching DiT, Vocos, Whisper и RUAccent (ударения)...')
!pip install -q f5-tts ruaccent openai-whisper soundfile torchaudio gradio
!apt-get install -y -qq ffmpeg
print('✅ Окружение готово!')


## 2. 🚀 Запуск премиум-студии
Нажмите **Play**. Через ~1.5 минуты появится ссылка **`Running on public URL: https://xxxx.gradio.live`**.
Откройте её на телефоне или компьютере.

In [ ]:
# @title 🚀 2. Запуск ElevenLabs Russian Studio (Launch WebUI)
import os
import json
import shutil
import torch
import gradio as gr
import tempfile
import soundfile as sf
import whisper
from ruaccent import RUAccent
from huggingface_hub import hf_hub_download
from f5_tts.model import DiT
from f5_tts.infer.utils_infer import load_model, load_vocoder, infer_process

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dev_name = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'
print(f'⚡ Устройство: {device} ({dev_name})')

VOICES_DIR = '/content/drive/MyDrive/ElevenLabs_Voices' if os.path.exists('/content/drive/MyDrive') else '/content/voices'
os.makedirs(VOICES_DIR, exist_ok=True)

print('📥 Загрузка нейровокодера Vocos 24kHz (чистый звук без робота)...')
vocoder = load_vocoder(vocoder_name='vocos', device=device)

print('📥 Загрузка русской модели F5-TTS Russian (5000+ часов русской речи)...')
ckpt_path = hf_hub_download(repo_id='Misha24-10/F5-TTS_RUSSIAN', filename='F5TTS_v1_Base_v2/model_last_inference.safetensors')
vocab_path = hf_hub_download(repo_id='Misha24-10/F5-TTS_RUSSIAN', filename='F5TTS_v1_Base/vocab.txt')
model_cfg = dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4)
model = load_model(DiT, model_cfg, ckpt_path=ckpt_path, vocab_file=vocab_path, device=device)

print('📥 Загрузка Whisper ASR (авто-расшифровка сэмплов)...')
asr = whisper.load_model('base', device=device)

print('📥 Инициализация RUAccent (автоматическая расстановка русских ударений)...')
accentizer = RUAccent()
accentizer.load(omograph_model_size='small', use_cuda=(device == 'cuda'))

print('🔥 Премиум-студия готова к генерации живой русской речи!')

def get_saved_voices():
    voices = []
    if os.path.exists(VOICES_DIR):
        for d in sorted(os.listdir(VOICES_DIR)):
            v_dir = os.path.join(VOICES_DIR, d)
            if os.path.isdir(v_dir) and os.path.exists(os.path.join(v_dir, 'sample.wav')):
                voices.append(d)
    return voices if voices else ['[Нет сохраненных голосов]']

def save_new_voice(voice_name, audio_path, custom_text):
    if not voice_name or not voice_name.strip():
        raise gr.Error('⚠️ Введите название голоса (например: Мой голос, Иван, Диктор)!')
    if not audio_path:
        raise gr.Error('⚠️ Загрузите или запишите аудио-образец голоса (5–15 секунд)!')
    
    clean_name = voice_name.strip().replace('/', '_').replace('\\', '_')
    voice_folder = os.path.join(VOICES_DIR, clean_name)
    os.makedirs(voice_folder, exist_ok=True)
    
    sample_wav = os.path.join(voice_folder, 'sample.wav')
    shutil.copyfile(audio_path, sample_wav)
    
    # Автоматическое распознавание текста через Whisper, если не указан
    if custom_text and custom_text.strip():
        text_raw = custom_text.strip()
    else:
        res = asr.transcribe(sample_wav, language='ru')
        text_raw = res.get('text', '').strip()
    
    if not text_raw:
        text_raw = 'Голосовой образец для клонирования речи.'
    
    text_accented = accentizer.process_all(text_raw)
    with open(os.path.join(voice_folder, 'ref_text.txt'), 'w', encoding='utf-8') as f:
        f.write(text_accented)
    
    new_list = get_saved_voices()
    status_msg = f'🎉 Голос "{clean_name}" успешно обучен и сохранен в библиотеку!'
    return (
        gr.update(choices=new_list, value=clean_name),
        gr.update(choices=new_list, value=clean_name),
        status_msg,
        sample_wav,
        text_raw
    )

def on_voice_select(selected_voice):
    if not selected_voice or selected_voice == '[Нет сохраненных голосов]':
        return None
    voice_wav = os.path.join(VOICES_DIR, selected_voice, 'sample.wav')
    return voice_wav if os.path.exists(voice_wav) else None

def generate_speech(selected_voice, text, speed):
    if not selected_voice or selected_voice == '[Нет сохраненных голосов]':
        raise gr.Error('⚠️ Выберите голос из списка или сначала добавьте новый во вкладке VoiceLab!')
    if not text or not text.strip():
        raise gr.Error('⚠️ Введите текст для озвучки!')
    
    voice_folder = os.path.join(VOICES_DIR, selected_voice)
    sample_wav = os.path.join(voice_folder, 'sample.wav')
    ref_txt_file = os.path.join(voice_folder, 'ref_text.txt')
    
    if not os.path.exists(sample_wav) or not os.path.exists(ref_txt_file):
        raise gr.Error(f'⚠️ Файлы голоса "{selected_voice}" повреждены или отсутствуют!')
    
    with open(ref_txt_file, 'r', encoding='utf-8') as f:
        ref_text_accented = f.read().strip()
    
    # Автоматическая расстановка ударений в целевом тексте
    target_accented = accentizer.process_all(text.strip())
    print(f'[SOTA TTS] Синтез речи: {target_accented[:60]}...')
    
    final_wave, final_sr, _ = infer_process(
        ref_audio=sample_wav,
        ref_text=ref_text_accented,
        gen_text=target_accented,
        model_obj=model,
        vocoder=vocoder,
        speed=float(speed),
        device=device
    )
    
    out_file = tempfile.mktemp(suffix='.wav')
    sf.write(out_file, final_wave, final_sr)
    return out_file

custom_css = '''
.gradio-container { font-family: 'Inter', sans-serif; max-width: 1050px !important; margin: auto; }
#btn-generate { background: linear-gradient(135deg, #f97316 0%, #ea580c 100%); color: white; font-weight: bold; font-size: 16px; border-radius: 8px; }
#btn-save-voice { background: linear-gradient(135deg, #10b981 0%, #059669 100%); color: white; font-weight: bold; font-size: 15px; border-radius: 8px; }
'''

initial_voices = get_saved_voices()

with gr.Blocks(title='ElevenLabs SOTA Russian Studio | Flow-Matching DiT', css=custom_css, theme=gr.themes.Soft(primary_hue='orange', neutral_hue='slate')) as demo:
    gr.Markdown(
        '''
        # 🎙️ ElevenLabs SOTA Russian Studio (Flow-Matching DiT)
        ### Ультра-премиум клонирование живого голоса на **русском языке** без металлического звука и роботов
        ---
        '''
    )
    
    with gr.Tabs():
        # --- ВКЛАДКА 1: ОЗВУЧКА СОХРАНЕННЫМ ГОЛОСОМ ---
        with gr.TabItem('⚡ 1. Озвучка текста (Studio)'):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown('### 👤 Выбор сохраненного голоса')
                    voice_dropdown = gr.Dropdown(
                        label='Сохраненный голос',
                        choices=initial_voices,
                        value=initial_voices[0],
                        interactive=True
                    )
                    voice_preview = gr.Audio(
                        label='Как звучит оригинал',
                        type='filepath',
                        interactive=False
                    )
                    voice_dropdown.change(fn=on_voice_select, inputs=[voice_dropdown], outputs=[voice_preview])
                    speed_slider = gr.Slider(label='Скорость речи (Speed)', minimum=0.7, maximum=1.4, value=1.0, step=0.05)
                    
                with gr.Column(scale=1):
                    gr.Markdown('### 📝 Текст для генерации')
                    text_input = gr.Textbox(
                        label='Введите текст на русском языке',
                        placeholder='Привет! Это абсолютно живой голос без роботизированного скрежета. Нейросеть с диффузией Flow-Matching сохраняет все мои интонации, паузы и тембр.',
                        lines=6
                    )
                    btn_gen = gr.Button('⚡ Озвучить выбранным голосом', elem_id='btn-generate', size='lg')
                    
                    gr.Markdown('### 🔊 Результат (Студийный 24kHz WAV)')
                    audio_output = gr.Audio(label='Готовая речь', type='filepath')
                    btn_gen.click(
                        fn=generate_speech,
                        inputs=[voice_dropdown, text_input, speed_slider],
                        outputs=[audio_output]
                    )

        # --- ВКЛАДКА 2: СОХРАНЕНИЕ / ОБУЧЕНИЕ НОВОГО ГОЛОСА ---
        with gr.TabItem('🎙️ 2. Добавить голос в библиотеку (VoiceLab)'):
            gr.Markdown(
                '''
                ### ➕ Создание нового профиля голоса
                Загрузите образец речи (5–15 секунд без музыки). Whisper автоматически расшифрует слова, а RUAccent расставит ударения для идеального совпадения!
                '''
            )
            with gr.Row():
                with gr.Column(scale=1):
                    new_voice_name = gr.Textbox(
                        label='Имя голоса (например: Мой голос, Павел Дуров, Диктор 1)',
                        placeholder='Например: Мой голос'
                    )
                    new_voice_audio = gr.Audio(
                        label='Аудио-образец голоса (5–15 секунд без шума и музыки)',
                        type='filepath',
                        sources=['upload', 'microphone']
                    )
                    new_voice_text = gr.Textbox(
                        label='Текст из аудио (оставьте пустым — Whisper сам расшифрует)',
                        placeholder='Оставьте пустым для авто-распознавания...',
                        lines=2
                    )
                    btn_save = gr.Button('💾 Обучить и сохранить голос в библиотеку', elem_id='btn-save-voice', size='lg')
                    
                with gr.Column(scale=1):
                    save_status = gr.Markdown('ℹ️ Голос еще не добавлен.')
                    saved_preview = gr.Audio(label='Сохраненный эталон', type='filepath', interactive=False)
                    recognized_text = gr.Textbox(label='Распознанный текст сэмпла', interactive=False)
                    
                    btn_save.click(
                        fn=save_new_voice,
                        inputs=[new_voice_name, new_voice_audio, new_voice_text],
                        outputs=[voice_dropdown, voice_dropdown, save_status, saved_preview, recognized_text]
                    )

print('👇 Публичная ссылка для телефона и ПК:')
demo.queue().launch(share=True, debug=False)
